In [1]:
#export 

import json
import logging
import pandas as pd
from typing import Any, Dict, Optional, Literal
import os
import math 
import ollama_manager
import re

PROMPT_ANALIZA_TODAS="analiza_todas_incidencias"
PROMPT_ANALIZA_CAUSA_RAIZ="analiza_incidencia_causa_raiz"
PROMPT_GENERA_PROPUESTAS="genera_propuestas"
PROMPT_GENERA_PROMPT="genera_prompt_app"
PROMPT_GENERA_DESC_SOL="calcula_descripcion_solucion"
PROMPT_CONSOLIDA_ANALISIS_GENERAL="consolida_analisis_general"

LISTA_CAMPOS_SPAGNA=["Número", "Descripción", "Descripción corta", "Notas de resolución", "Notas de trabajo", "Estado", "Estado del incidente"]

# Tipos de prompt permitidos: mismos nombres que los métodos "vacíos"
PromptTipo = Literal[
    PROMPT_ANALIZA_TODAS,
    PROMPT_ANALIZA_CAUSA_RAIZ,
    PROMPT_GENERA_PROPUESTAS,
    PROMPT_GENERA_PROMPT,
    PROMPT_GENERA_DESC_SOL,
    PROMPT_CONSOLIDA_ANALISIS_GENERAL
]

""" 
Variables internas que genera la clase:
    * _app_name -> Nombre de la aplicación con la que estamos trabajando. 
    * _fichero_incidencias -> Path + Nombre del fichero de incidencias. 
    * _apps -> Listado de apps configuradas en el fichero de apps.ini.
    * _promtps -> Listado de prompts configurados en el fichero de prompts.ini.
    * 

Métodos internos:
    * _setPrompt -> Sirve para asignar un prompt leído del ficheor 
"""


/Users/zzddfge/.pyenv/versions/mlx-31211/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
objc[12468]: Class AVFFrameReceiver is implemented in both /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x12461c3a8) and /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x13e5e03a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[12468]: Class AVFAudioReceiver is implemented in both /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x12461c3f8) and /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-pac

' \nVariables internas que genera la clase:\n    * _app_name -> Nombre de la aplicación con la que estamos trabajando. \n    * _fichero_incidencias -> Path + Nombre del fichero de incidencias. \n    * _apps -> Listado de apps configuradas en el fichero de apps.ini.\n    * _promtps -> Listado de prompts configurados en el fichero de prompts.ini.\n    * \n\nMétodos internos:\n    * _setPrompt -> Sirve para asignar un prompt leído del ficheor \n'

In [2]:
#export

############################################################
############################################################
############################################################
## CLASE EXCEPCIÓN DE INCIDENCIAS

In [3]:
#export

class IncidenciaException(Exception):
    """
    Excepción de dominio para tratar incidencias localmente.
    Úsala para capturarla en el mismo módulo/capa y decidir la acción
    (reintento, degradación, log, mensaje al usuario, etc.).
    """
    def __init__(self, mensaje: str | None = None, causa: Exception | None = None):
        super().__init__(mensaje if mensaje is not None else "")
        self.causa = causa  # opcional: guarda la excepción original

In [4]:
#export

############################################################
############################################################
############################################################
## CLASE PARAMETROS DE INCIDENCIAS

In [5]:
#export

class ParametrosIncidencias:
    def __init__(self, fichero_apps: str = "parametros/app.ini", fichero_prompts: str = "parametros/prompts.ini") -> None:
        """
        """
        # Diccionarios internos
        # Ejemplo de apps: {"app1": "aplicación 1", ...}
        self._apps: Dict[str, str] = {}

        # Ejemplo de prompts: {"app1": { "analiza_todas_incidencias": "...", ... }, ...}
        self._prompts: Dict[str, Dict[str, str]] = {}

        # Carga inicial
        self._loadApp(fichero_apps)
        self._loadPrompts(fichero_prompts)

    _clase="ParametrosIncidencias"

    # =========================
    # Métodos privados: Apps
    # =========================
    def _setApp(self, nombre_app: str, descripcion_app: str) -> None:
        """
        MÉTODO PRIVADO.
        Añade/actualiza una app en el diccionario interno de apps.
        Ej: _setApp("app1", "Aplicación 1")
        """
        method="_setApp"
        logging.debug(f"{self._clase}.{method}-START nombre_app={nombre_app} descripcion_app={descripcion_app}")
        self._apps[nombre_app] = descripcion_app

    def _getApp(self, nombre_app: str) -> str:
        """
        MÉTODO PRIVADO.
        Devuelve la descripción de una app concreta.
        Si la app no existe, lanza un KeyError.
        """
        method="_getApp"

        logging.debug(f"{self._clase}.{method}-START ombre_app={ombre_app}")

        if nombre_app not in self._apps:
            logging.error(f"{self._clase}.{method}-ERROR App {nombre_app} no encontrada en el diccionario de apps")
            raise KeyError(f"{self._clase}.{method}-ERROR La app '{nombre_app}' no existe en app.ini")
        return self._apps[nombre_app]

    # =========================
    # Métodos privados: Prompts
    # =========================
    def _setPrompt(self, nombre_app: str, tipo_prompt: PromptTipo, texto_prompt: str) -> None:
        """
        MÉTODO PRIVADO.
        Añade/actualiza un prompt para una app concreta.
        El tipo de prompt está restringido a PromptTipo.
        """
        method="_setPrompt"
        logging.debug(f"{self._clase}.{method}-START nombre_app={nombre_app}, tipo_prompt={tipo_prompt}, texto_prompt={texto_prompt}")

        if nombre_app!="generico" and nombre_app not in self._apps:
            logging.error(f"{self._clase}.{method}-ERROR App {nombre_app} no encontrada en el diccionario de apps")
            raise KeyError(f"{self._clase}.{method}-ERROR La app '{nombre_app}' no existe en app.ini")
        
        if nombre_app not in self._prompts:
            self._prompts[nombre_app] = {}
        self._prompts[nombre_app][tipo_prompt] = texto_prompt

    # =========================
    # Métodos privados: Carga de ficheros
    # =========================
    def _loadApp(self, ruta_fichero: str) -> None:
        """
        MÉTODO PRIVADO.
            Lee el fichero (una línea = un JSON) y carga las apps usando _setApp.

        FORMATO ESPERADO (JSON Lines):
            {"nombre_app":"aplicación 1", "descripcion_app":"descripcion 1", "servicenow_id":"asadkadalda"}
            {"nombre_app":"aplicación 2", "descripcion_app":"descripcion 2", "servicenow_id":"asadkadalda"}

        Es decir, cada línea debe ser un objeto JSON con las claves:
        - nombre_app (str)
        - descripcion_app (str)
        - servicenow_id (str)   # si no lo usas en _setApp, igualmente se valida/queda disponible
        """
        method = "_loadApp"
        logging.debug(f"{self._clase}.{method}-START Cargando apps desde fichero: {ruta_fichero}")

        if not os.path.exists(ruta_fichero):
            msg = f"{self._clase}.{method}-ERROR {ruta_fichero} no existe."
            raise IncidenciaException(msg)

        try:
            df = pd.read_json(ruta_fichero, lines=True)
        except ValueError as e:
            raise IncidenciaException(f"{self._clase}.{method}-ERROR JSON inválido en {ruta_fichero}", causa=e) from e

        logging.debug(f"{method}- Número app leídas son: {len(df)}")

        if df.empty:
            logging.debug(f"{self._clase}.{method}-INFO El fichero {ruta_fichero} está vacío. No se cargan apps.")
            return

        # Validación de columnas esperadas
        requeridas = {"nombre_app", "descripcion_app", "servicenow_id"}
        faltan = requeridas - set(df.columns)
        if faltan:
            raise IncidenciaException(
                f"{self._clase}.{method}-ERROR Formato inválido en {ruta_fichero}: faltan claves {sorted(faltan)}"
            )

        # Iteración por registros
        rows = df.to_dict(orient="records")

        for i, row in enumerate(rows, start=1):
            nombre_app = row.get("nombre_app")
            descripcion_app = row.get("descripcion_app")
            servicenow_id = row.get("servicenow_id")

            # Control de nulos/NaN y tipos
            if not isinstance(nombre_app, str) or not nombre_app.strip():
                raise IncidenciaException(f"{self._clase}.{method}-ERROR Línea {i}: 'nombre_app' debe ser string no vacío.")
            if not isinstance(descripcion_app, str) or not descripcion_app.strip():
                raise IncidenciaException(f"{self._clase}.{method}-ERROR Línea {i}: 'descripcion_app' debe ser string no vacío.")
            if not isinstance(servicenow_id, str) or not servicenow_id.strip():
                raise IncidenciaException(f"{self._clase}.{method}-ERROR Línea {i}: 'servicenow_id' debe ser string no vacío.")

            # Si tu _setApp NO acepta servicenow_id, lo ignoras (o lo guardas en otro sitio).
            # Si tu _setApp SÍ lo acepta, cambia a: self._setApp(nombre_app, descripcion_app, servicenow_id)
            self._setApp(nombre_app.strip(), descripcion_app.strip())
    
    def _loadPrompts(self, ruta_fichero: str) -> None:
        """
        MÉTODO PRIVADO.
        Lee el fichero prompts.ini (o el que se indique) y carga los prompts usando _setPrompt.

        FORMATO SOPORTADO (único formato):
        {
            "nombre_app": {
                "analiza_todas_incidencias": "....",
                "analiza_incidencia_causa_raiz": "....",
                "genera_propuestas": "....",
                "genera_prompt_app": "....",
                "calcula_descripcion_solucion": "...."
            },
            "generico": {
                ...
            }
        }

        - Las claves de primer nivel deben coincidir con las apps definidas en app.ini,
        EXCEPTO "generico", que se permite siempre.
        """
        method = "_loadPrompts"
        logging.debug(f"{self._clase}.{method}-START Cargando prompts desde fichero: {ruta_fichero}")

        if not os.path.exists(ruta_fichero):
            raise IncidenciaException(f"{self._clase}.{method}-ERROR {ruta_fichero} no existe.")

        # Leer JSON (un único objeto en todo el fichero)
        try:
            with open(ruta_fichero, "r", encoding="utf-8") as f:
                contenido = f.read().strip()
                if not contenido:
                    raise IncidenciaException(f"{self._clase}.{method}-INFO El fichero {ruta_fichero} está vacío. No se cargan prompts.")
                data = json.loads(contenido)
        except json.JSONDecodeError as e:
            raise IncidenciaException(f"{self._clase}.{method}-ERROR JSON inválido en {ruta_fichero}", causa=e) from e
        except OSError as e:
            raise IncidenciaException(f"{self._clase}.{method}-ERROR No se pudo leer {ruta_fichero}", causa=e) from e

        if not isinstance(data, dict):
            raise IncidenciaException(f"{self._clase}.{method}-ERROR El fichero {ruta_fichero} debe contener un único objeto JSON de nivel superior (dict).")

        # Validación dinámica coherente con PromptTipo
        tipos_permitidos: set[str] = set(PromptTipo.__args__)  # type: ignore[attr-defined]

        for nombre_app, prompts_app in data.items():
            logging.debug(f"{self._clase}.{method}- Procesando prompts para la app: {nombre_app}")

            if not isinstance(nombre_app, str):
                raise ValueError("Las claves de primer nivel en prompts.ini deben ser cadenas.")

            # Comprobamos que la app exista en _apps, pero permitimos 'generico'
            if self._apps and nombre_app != "generico" and nombre_app not in self._apps:
                msg=f"{self._clase}.{method} La app {nombre_app} en {ruta_fichero} no existe en el diccionario de apps y no es 'generico'"
                logging.debug(msg)
                raise ValueError(msg)

            if not isinstance(prompts_app, dict):
                raise ValueError(f"{self._clase}.{method}El valor de '{nombre_app}' en {ruta_fichero} debe ser un diccionario con los distintos tipos de prompt.")

            for tipo, texto in prompts_app.items():
                logging.debug(f"{self._clase}.{method}- Procesando tipo de prompt '%s' para app {tipo, nombre_app}")

                if not isinstance(tipo, str) or not isinstance(texto, str):
                    raise ValueError(f"{self._clase}.{method} En prompts.ini, para la app '{nombre_app}', cada clave de prompt y su valor deben ser cadenas.")

                if tipo not in tipos_permitidos:
                    raise ValueError(
                        f"Tipo de prompt '{tipo}' no permitido para la app '{nombre_app}'. "
                        f"Debe ser uno de: {sorted(tipos_permitidos)}"
                    )

                # Cast a PromptTipo para el type checker
                tipo_prompt_literal: PromptTipo = tipo  # type: ignore[assignment]
                self._setPrompt(nombre_app, tipo_prompt_literal, texto)


In [6]:
############################################################
############################################################
############################################################
## CLASE INCIDENCIAS

In [7]:
#export

class Incidencias:
    _clase="Incidencias"

    def __init__(self, app:str, fichero_incidencias:str, fichero_apps: str = "parametros/app.ini", fichero_prompts: str = "parametros/prompts.ini") -> None:
        """
        Constructor: inicializa diccionarios y carga apps y prompts desde los ficheros.
        """
        method="___init___"

        logging.debug(f"{self._clase}.{method}-START Incidencias con fichero_apps={fichero_apps}, fichero_prompts={fichero_prompts}")

        # ----
        # Inicialización de variables internas
        # ----

        #Nombre de la App que vamos a procesar.

        # Nombre del fichero con incidencias y App que va a gestioanr.
        # Preparamos para que si el fichero de incidencias no acaba en .xls se lo añadimos para mayor flexibilidad.
        self._app_name = app
        self._fichero_incidencias = fichero_incidencias if fichero_incidencias.lower().endswith(".xlsx") else f"{fichero_incidencias}.xlsx"
        if not os.path.exists(self._fichero_incidencias):
            raise IncidenciaException(f"{self._clase}.{method}-ERROR No existe el fichero {self._fichero_incidencias}")
        
        logging.debug(f"{self._clase}.{method}- Inicializado para {self._app_name} y fichero {self._fichero_incidencias}")

        #Parámetros de carga de las apps soportadas y de los prompts soportados.
        parametros = ParametrosIncidencias(fichero_apps=fichero_apps, fichero_prompts=fichero_prompts)
        self._apps = parametros._apps 
        self._prompts = parametros._prompts

        #Cargamos las incidencias. 
        self._df = self._loadIncidencias()

        #Preparamos el diccionario para tener los dfs de cada modelo, en caso se usen con una lista de modelos.
        self._df_modelos = {}

        #Por defecto, timeout de 5 minutos.
        self._timeout=300.0

    # =========================
    # Métodos Auxiliares
    # =========================

    def _get_tokens(self, prompt:str):
        palabras = prompt.split()
        tokens=math.ceil(len(palabras)*1.35)

        return palabras, tokens

    def _renderiza_prompt(self, row, lista_variables: list, lista_campos:list, prompt: str) -> str:
        ctx = {}
        for var, col in zip(lista_variables, lista_campos):
            val = row[col]
            if pd.isna(val):
                val = ""
            ctx[var] = val

        return prompt.format_map(ctx)

    def _ejecuta_prompts(self, prompt:str, model:str):
        method="_ejecutaPrompts"

        system = ''' 
            You are an expert analist in detecting the root cause for incidents in the department of micro computer. 
            You works for a very big company first of its class. 
            You are the best detecting root cause and proposing viable solutions. 
            You have a reward of 2.000.000$ for reducing number of incidents with your solutions. 
            If cant make proposals, you are fired. 
            The only lenguage you can answered is spanish whatever language you can find in the incidents. 
        '''

        palabras, num_tokens = self._get_tokens(prompt)
        logging.debug(f"{method}-START ejecutando modelo {model}, temperatura={self._temperature} para el prompt={prompt}, timeout={self._timeout}")
    
        resp = ollama_manager.call_ollama_api(model=model, system_message=system, user_message=prompt, temperature=self._temperature, timeout=self._timeout)
        resp = resp.lower().replace("json", '').replace("```", '')
        content, cot = ollama_manager.separa_thinking_contenido(resp)
    
        return content, cot, num_tokens


    def _ejecutarAnalisisGeneral(self, model:str, listado_json:list, prompt:str, sufix_file:str):
        """
        The function `_ejecutarAnalisisGeneral` is a helper function that calls the LLM with the incident description / solution. The steps are:
        - Invoking LLM
        - Saving all the response in file. The file is the original file + sufix with the model and the execution. Sufix is already calculated in the invocation.
        
        Parameters:
            model:str --> Modelo actual.
            listado_json:list --> Lista con los objetos de json.
            prompt:str --> Prompt, que lleva dos variables, listado_json y app. 
            sufix_file:str --> Sufijo del fichero que vamos a generar de forma individuall.

        Exception:
            None.

        Return:
            llm_ouput: str --> Análisis del LLM. No considera la cadena de thinking. 
        """
        method="_ejecutarAnalisisGeneral"

        #2.- Miramos si el fichero está creado. 
        fichero_ouput=f"{self._fichero_incidencias.replace('.xlsx','')}_{sufix_file}.txt"

        #Si ya existe no hacemos nada.
        if os.path.exists(fichero_ouput):
            logging.info(f"{self._clase}.{method}-START fichero {fichero_ouput} ya existe, no se genera.")

            with open(fichero_ouput, "r",  encoding="utf-8") as f:
                todo = f.read()
                todo = todo.lstrip("\ufeff")  # por si hay BOM
                m = re.match(r"^\s*<think>.*?</think>\s*", todo, flags=re.DOTALL | re.IGNORECASE)

                salida = todo[m.end():] if m else todo
                logging.debug(f"{self._clase}.{method}- salida recuperada={salida}")
        else:
            #2.- Ejecutamos el prompt
            json_object = {"analisis_list": listado_json}
            json_output = json.dumps(json_object, indent=2)

            prompt_real=prompt.format(listado_json=str(json_output), app=self._app_name, numero_incidentes=len(listado_json))
            content, cot, num_tokens = self._ejecuta_prompts(prompt_real, model)
        
            #Convertimos salida en texto, y cambiamos el \n por \r\n.
            salida_cot = f"{cot}".replace('\r\n', '\n').replace('\r', '\n').replace('\\n', '\n')
            salida = f"{content}".replace('\r\n', '\n').replace('\r', '\n').replace('\\n', '\n')

            #Genera un archivo de salida Excel con el nombre del modelo, reemplazando el : por _.
            with open(fichero_ouput, "w",  encoding="utf-8") as f:
                f.write(f"<think>\n{salida_cot}\n</think>\n{salida}")
            logging.info(f"{method}-END fichero {fichero_ouput} generado.")

        return fichero_ouput, salida

    def _obtenerDatosIncidencias(self, df_proceso: pd.DataFrame, numero_elemento_inicial: int, context_size: int, lista_campos_variables: list, margen=0.9):
        """
        The function `_obtenerDatosIncidencias` is a helper function that processes the DataFrame e prepares the list
        of json elements to send the LLM. It considers not preparing more elements that those can enter in the 
        context window we already have.  
        
        Parameters:
            df_proceso: pd.DataFrame --> DataFrame con los datos de incidencias
            numero_elemento_inicial:int --> Número del primer elemento a procesar
            context_size:int --> Tamaño máximo de contexto
            lista_campos_variables:list --> Lista de campos a incluir en el JSON
            margen:numeric --> Margen que queremos mantener para asegurar que no nos pasamos del contexto. Por defecto, 0.9
        
        Exception:
            None.

        Return:
            numero_elemento_final --> last element processed
            lista_objetos_json --> processed json element.
        """
        method="_obtenerDatosIncidencias"

        lista_objetos_json = []
        numero_elemento_actual = numero_elemento_inicial
        tokens_totales = 0
        limite_tokens = int(context_size * margen)  # 10% de margen
        continuar = True 

        logging.debug(f"{self._clase}.{method}-START Entrando con={numero_elemento_inicial} / {len(df_proceso)-1}, context_size={context_size}")
    
        while numero_elemento_actual < len(df_proceso) and continuar:
            # Recuperar una fila del dataframe
            row = df_proceso.iloc[numero_elemento_actual]
        
            # Crear objeto JSON con los campos especificados
            objeto_json = {}
            objeto_json["id"] = row.iloc[0]
            for campo in lista_campos_variables:
                objeto_json[campo] = row.get(campo, "")
        
            # Verificar la longitud en tokens
            _, tokens_row = self._get_tokens(str(objeto_json))
            logging.debug(f"{self._clase}.{method}- Elemento={numero_elemento_actual}-Id={row.iloc[0]}, tokens={tokens_row}, aggregado={tokens_totales}")
        
            # Verificar si se puede agregar el elemento
            if tokens_totales + tokens_row <= limite_tokens:
                lista_objetos_json.append(objeto_json)
                tokens_totales += tokens_row
                numero_elemento_actual += 1
            else:
                # Salir del bucle si excede el límite
                continuar = False
    
        return numero_elemento_actual, lista_objetos_json

    # =========================
    # Métodos Privados 
    # =========================

    def _loadIncidencias(self, lista_campos = LISTA_CAMPOS_SPAGNA):
        """
        Lee un archivo de incidencias Excel y se queda solo con el listado de campos que se especifican. 

        Este debe ser el orden de los campos. 

        Sólo da libertad para poder cargar diferentes excels que tengan nombre de campos distintos de forma paramétrica.

        Parametros:
            - lista_campos: Lista de campos que se quieren extraer del archivo Excel. Por defecto, se extraen los campos "Número", "Descripción corta" y "Notas de resolución".
    
        Return:
            Un DataFrame con los campos especificados extraídos del archivo Excel.
        """
        method ="_loadIncidencias"

        file_name_path = self._fichero_incidencias
        logging.info(f"{self._clase}.{method}-START - Leyendo el archivo Excel: {file_name_path}")

        df = pd.read_excel(file_name_path, index_col=None)
        logging.info(f"{self._clase}.{method}- {df.shape[0]} registros leídos.")

        #Copia y selección de los tres campos que necesitamos.
        df_filtrado = df.copy()
        #df_filtrado = df_filtrado[lista_campos]
        df_filtrado.columns = df_filtrado.columns.str.lower()
        #df_filtrado.rename(columns={lista_campos[0].lower(): "id"
            #                    , lista_campos[1].lower(): "desc"
            #                    , lista_campos[2].lower(): "res"
            #                    }, inplace=True)

        logging.debug(f"{self._clase}.{method}-END - Se ha filtrado el DataFrame para obtener los campos especificados: {df_filtrado.columns}")

        return df_filtrado
    
    ##
    ## Necesarios para trabajar con los DFs de cada modelo. 
    ##

    def _setDf(self, modelo:str, df: pd.DataFrame) -> None:
        """
        Asigna un df en el diccionario de dfs para cada modelo. 
        
        :param self: Descripción
        :param modelo: el nombre del modelo de ia. 
        """
        method="_setDf"
        self._df_modelos[modelo] = df

    def _getDf(self, modelo:str) -> pd.DataFrame:
        """
        Recupera el df de un modelo.
        
        :param self: Descripción
        :param modelo: nombre del modelo
        :return: Datos del Df que se han procesado con ese modelo, o None en caso contrario.
        :rtype: DataFrame
        """
        method="_getDf"

        df = self._df_modelos.get(modelo)
        df_return = None
        if df is None:
            logging.debug(f"{method}-ERROR no se ha procesado aún datos para el modelo {modelo}")
        else:
            df_return = df.copy()
        
        return df_return
    ##
    ## Necesarios para trabajar con promts.
    ## 

    def _getPrompt(self, nombre_app: str, tipo_prompt: str) -> str:
        """
        MÉTODO PRIVADO.
        Devuelve el texto de un prompt para una app y tipo de prompt concreto.
        - nombre_app: clave de la app (por ejemplo, "app1")
        - tipo_prompt: uno de los valores de PromptTipo

        Comportamiento:
        1) Intenta recuperar el prompt específico de la app.
        2) Si no existe, intenta recuperar el prompt de la app 'generico'.
        3) Si tampoco existe, lanza KeyError.
        """
        method="_getPrompt"
        logging.debug(f"{self._clase}.{method}-START nombre_app={nombre_app}, tipo_prompt={tipo_prompt}")

        prompt=None
        # 1) Buscar prompt específico de la app
        if nombre_app in self._prompts:
            prompts_app = self._prompts[nombre_app]
            if tipo_prompt in prompts_app:
                logging.debug(f"{self._clase}.{method} Encontrado prompt específico para app {nombre_app} y tipoPromtp={tipo_prompt}")
                prompt=prompts_app[tipo_prompt]

        # 2) Fallback a la app genérica
        if prompt is None and "generico" in self._prompts:  # <-- CAMBIO generico
            logging.debug(f"{self._clase}.{method}- Cambiando a genérico para app={nombre_app} y tipoPrompt={tipo_prompt}")
            prompts_generico = self._prompts["generico"]
            if tipo_prompt in prompts_generico:
                logging.info(f"{self._clase}.{method}- Encontrado.")
                prompt=prompts_generico[tipo_prompt]

        # 3) Error si no existe ni específico ni genérico
        if prompt is None:
            msg =f"{self._clase}.{method}-ERROR No existe para app {nombre_app} el tipoPrompt={tipo_prompt} ni en la app 'generico'"
            logging.error(msg)
            raise KeyError(msg)
    
        return prompt
    
    ##
    ## Necesarios para procesar el excel 
    ##

    def _addColumns(self, df_nuevo, sufijo, fichero_output_path, model, prompt, lista_variables, lista_campos):
        """ 
        Agrega columnas a un DataFrame llamado df con las causas raíz y clasificaciones obtenidas por el modelo Ollama.
    
        Parametros:
            df (pandas.DataFrame): El DataFrame que contiene las descripciones y notas de resolución.
            sufijo (str) : Sufijo para las columnas de status y cot porque puede haber diversos prompts que creen información extra en el excel.
            fichero_output_path (str): Nombre del fichero sobre el cual se está grabando cada 20 iteracciones.
            model (str): El nombre del modelo Ollama a usar para obtener las causas raíz y clasificaciones. Por defecto, es "gemma3:latest".
            prompt (str): Prompt que se va a ejecutar.

        Return:
            Una copia del data frame original con las nuevas columnas para todas las incidencias.
        """
        method="add_columns"

        logging.info(f"{method}-START iniciando generación información para {df_nuevo.shape[0]} registos con el modelo {model} y cot {self._thinking}")
        logging.debug(f"{method}- lista_variables:{lista_variables}, lista_campos:{lista_campos}")
        
        generado=0
        logging.debug(f"{method}- columnas del DF: {df_nuevo.columns}")
        for index, row in df_nuevo.iterrows():
            #Generamos un bucle para calcular en tiempo de ejecución 
            prompt_modificado =  self._renderiza_prompt(row, lista_variables, lista_campos, prompt)

            status = row[f'status_{sufijo}']

            #Sólo procesamos los datos con la IA si no están ya generados.
            if status != 'OK':
                #Bucle para volver a pedir cuando haya un problema en el token devuelto.
                while status != "OK":
                    content, cot, num_tokens = self._ejecuta_prompts(prompt_modificado, model)
                    logging.info(f"{method}- Index={index}, ID = {row.iloc[0]} con {num_tokens} tokens --> output={content}")
                    logging.debug(f"{method}- Index={index}, Longitud CoT={len(cot)}, longitud Content={len(content)}")
                    
                    try:
                        contenido = json.loads(content)
                        logging.debug(f"{method}- generado contenido={contenido}")
                        
                        #Procesamos el json para meter de forma dinámica las columnas.
                        num=0
                        for key, value in contenido.items():
                            col = key.lower()

                            # crea la columna si no existe (opcional; pandas la crea igualmente al asignar)
                            if col not in df_nuevo.columns:
                                df_nuevo[col] = None

                            df_nuevo.at[index, col] = value
                            num+=1
                            
                        logging.debug(f"{method}- procesados {num} campos")

                        df_nuevo.at[index, f'cot_{sufijo}'] = cot[:30000]
                        df_nuevo.at[index, f'status_{sufijo}'] = "OK"

                        #MArcamos la opción de salir del while.
                        status = 'OK'
                    except Exception as e:
                        logging.error(f"{method}-ERROR al parsear JSON (ID={row.iloc[0]}) con error {e}")
                        df_nuevo.at[index, f'status_{sufijo}'] = "ERROR"
            
                #Cada 50 interacciones guardamos el fichero para ir manteniendo el progreso.
                generado+=1
                if generado % 20 == 0:
                    df_nuevo.to_excel(fichero_output_path, index=False)
                    logging.info(f"{method}- Guardando archivo Excel en {fichero_output_path}")

            else:
                logging.debug(f"{method}- Index={index}, Status = OK para el id {row.iloc[0]}")
            

        return df_nuevo
    
    def _procesaIncidencias(self, df, sufijo, model, prompt, lista_variables, lista_campos):
        """ 
        Procesa las incidencias una a una. Recibe un prompt que generará un json. Se debe procesar el prompt 
        para guardar los campos del json en el dataframe.

        """
        method="procesa_incidencias"

        #Preparamos el nombre del modelo.
        model_name = model.replace(':', '_')

        #Generamos el nombre del fichero de salida.
        fichero_ouput_path=f"{self._fichero_incidencias.replace('.xlsx','')}_{model_name}.xlsx"

        #Si el fichero existe, lo cargamos en vez de procesar los datos que nos llegan. 
        #Si no, procesamos los datos que nos llegan. 
        if os.path.exists(fichero_ouput_path):
            df_nuevo = pd.read_excel(fichero_ouput_path, index_col=None)
            logging.info(f"{method}-START Fichero {fichero_ouput_path} ya existente. Se ha leído.")
        else:
            df_nuevo = df
            logging.info(f"{method}-START Procesando por primera vez al no existir {fichero_ouput_path}.")
    
        #Procesa cada incidencia.
        df_nuevo = self._addColumns(df_nuevo, sufijo, fichero_ouput_path, model, prompt, lista_variables, lista_campos)

        #Genera un archivo de salida Excel con el nombre del modelo, reemplazando el : por _.
        df_nuevo.to_excel(fichero_ouput_path, index=False)
        self._setDf(model_name, df_nuevo.copy())
    
        logging.info(f"{method}- Archivo de salida generado: {fichero_ouput_path}")

        return df_nuevo

    def _consolidaAnalisisGeneral(self, model, fichero_ouput:str, lista_ficheros:list, lista_analisis:list) -> str:
        """
        La función `_consolidaAnalisisGeneral` consolida en un único fichero todas las causas raices que se han analizado 
        respetando la ventana de contexto existente. 

        Parámetros:
            - fichero_output: str --> fichero donde se debe consolidar todos los ficheros que se han generado. 
            - lista_ficheros: list --> Lista de ficheros que se han generado. 
            - salida: list --> Lista de contenidos que se han generado. 
        
        Exception:
            None
        
        Return: 
            - salida: str --> La salida que se ha consolidado. 
        """
        method="_consolidaAnalisisGeneral"

        logging.info(f"{self._clase}.{method}-START consolidando todos los análisis (un total de {len(lista_analisis)})")

        #Si hemos tenido que ejecutar más de 1 vez, tenemos que resumir
        if len(lista_analisis) > 1:
            #consolidamos los ficheros que se han generado. Se crea un nuevo fichero y se borran el resto.
            json_dict = {f"exec_{i}": var for i, var in enumerate(lista_analisis)}
            json_output = json.dumps(json_dict)

            #Recuperamos prompt y ejecutamos. 
            prompt = self._getPrompt(self._app_name, PROMPT_CONSOLIDA_ANALISIS_GENERAL)
            prompt_with_data = prompt.format(json_output=json_output)
            content = ollama_manager.call_ollama_api(model, "", prompt_with_data)
            
            #grabamos a disco
            with open(fichero_ouput, "w",  encoding="utf-8") as f:
                f.write(content)

            #eliminamos los fihceros temporales.
            """ 
            for file in lista_ficheros:
                try:
                    os.remove(file)
                    logging.debug(f"{self._clase}.{method}: Borrando {file}")
                except FileNotFoundError:
                    logging.debug(f"{self._clase}.{method}: No se encuentra {file}")
                except PermissionError:
                    logging.debug(f"{self._clase}.{method}: Permiso denegado al borrar {file}")
            """
        else:
            #Tenemos que renombar el fichero sin el ejecución_1
            fichero_origen=f"{self._fichero_incidencias.replace('.xlsx','')}_{model.replace(':', '_')}_ejecucion_1.txt"
            os.rename(fichero_origen, fichero_ouput)


    # =========================
    # Métodos Públicos
    # =========================

    def setModeloEntreFases(self, modelo:str):
        method="setModeloEntreFases"
        logging.info(f"{modelo}-START Asignado intra modelo {modelo} para los modelos configurados {self._modelos}")
        if modelo not in self._modelos:
            raise IncidenciaException(f"{method}-ERROR este modelo no está configurado en los modelos actuales.")
        self._modelo_entre_modelos = modelo
        
    def setDatosModelos(self, modelos:list, temperature: float, flg_thinking:bool) -> None: 
        self._modelos=modelos
        self._temperature = temperature
        self._thinking = flg_thinking

    def setTimeout(self, timeout=int):
        self._timeout = timeout

    def getApps(self):
        return list(self._apps.keys())

    #######
    # USO DE LLM
    #######

    #######
    # Métodos que tratan el total de las incidencias, si permite el contexto.

    def analiza_todas_incidencias(self, context_size:int) -> None:
        """
        The function `analiza_todas_incidencias` is the second method to use in the workflow of incident analysis. 
        
        It uses the already calculated desc, res fields to generates a first global classification of the incidents for a given application.  

        It has the following considerations:
        - Only processes incidents that are already closed, so it can have description and also solutions informed. 
        - it considers the context size given, so it sends to llm information only considering the context window. 
        - If it has been to make more than one call to the llm, it makes a new call to consolidate all classifications into one only file.

        Parameters:
            context_size:integer --> The name of the fields that we need use from the data frame loaded. First. field must be the ID for logging reasons.

        Exception:
            Raises an IncidenException if validations are not superated. Size of context_size, it hasn't be assigned the models, etc.

        Return:
            None
        """
        method="analiza_todas_incidencias"
        logging.info(f"{self._clase}.{method}--------------------------------------------------------------")
        logging.info(f"{self._clase}.{method}-START Inicio análisis y clasificación general de incidencias.")
        logging.info(f"{self._clase}.{method}- Análisis general sobre el dataframe.")
        logging.info(f"{self._clase}.{method}- modelos a usar: {self._modelos}, razonamiento para modelos generales: {self._thinking}")
        logging.info(f"{self._clase}.{method}- temperatura: {self._temperature}, aplicación: {self._app_name}")
        logging.info(f"{self._clase}.{method}--------------------------------------------------------------")

        prompt = self._getPrompt(self._app_name, PROMPT_ANALIZA_TODAS)
        logging.debug(f"{self._clase}.{method}- Prompt a usar: ")

        ### 
        #Validamos que recibimos todos los campos y preparamos la lista de campos que están en el prompt
        #Estas variables se tienen que generar en tiempo de ejecución para cada incidencia, por eso se tienen que pasar al método _procesaIncidencia.
        if context_size < 2028 :
            raise IncidenciaException(f"{method}-ERROR Tamaño de contexto insuficiente.")

        if self._modelos is None:
            raise IncidenciaException(f"{method}-ERROR No se han especificado los modelos a usar.")
        ### 

        #Partimos del último modelo generado.
        model = self._modelo_entre_modelos
        df_proceso = self._getDf(model)
        if df_proceso is None:
            raise IncidenciaException(f"{method}-ERROR El modelo {model} no ha sido usado para generar las variables `desc` con el resumen de descripción y `res` con el resumen de resolución.")

        logging.info(f"{self._clase}.{method}-START usando datos del modelo {model}")
        lista_campos_variables=["desc", "res"]
        lista_salida = []
        
        for model in self._modelos:
            logging.info(f"{self._clase}.{method}- iniciando proceso para el modelo {model}")

            #Copiamos el DF.
            numero_elemento = -1
            ejecuciones = 0
            lista_analisis = []
            lista_ficheros = []

            #Si ya está creado el fichero, no lo procesamos el fichero.
            fichero_ouput=f"{self._fichero_incidencias.replace('.xlsx','')}_{model.replace(':', '_')}.txt"

            if os.path.exists(fichero_ouput):
                logging.info(f"{self._clase}.{method}- ya existe el fichero {fichero_ouput} para el modelo {model}.")
                with open(fichero_ouput, "r",  encoding="utf-8") as f:
                    todo = f.read()
                    todo = todo.lstrip("\ufeff")  # por si hay BOM
                    m = re.match(r"^\s*<think>.*?</think>\s*", todo, flags=re.DOTALL | re.IGNORECASE)
                    salida = todo[m.end():] if m else todo
                    logging.debug(f"{self._clase}.{method}- salida recuperada={salida}")
                    lista_salida.append(salida)
                continue

            #Vamos procesando en bloques de context_size.
            while numero_elemento < len(df_proceso):
                ejecuciones+=1
                numero_elemento, listado_json = self._obtenerDatosIncidencias(df_proceso, numero_elemento+1, context_size, lista_campos_variables)
                _, tokens = self._get_tokens(str(listado_json))
                logging.info(f"{self._clase}.{method}-  Ejecucion={ejecuciones}, procesando {numero_elemento}/{len(df_proceso)} elementos con un total de {tokens} tokens.")
                #logging.info(f"{self._clase}.{method}-  Último código de incidencia {df_proceso.iloc[numero_elemento][0]}")

                sufix_file = model.replace(':', '_') + "_" + f"ejecucion_{ejecuciones}"
                fichero_ejecucion_output, salida = self._ejecutarAnalisisGeneral(model, listado_json, prompt, sufix_file)
                logging.info(f"{self._clase}.{method}-  Ejecucion {ejecuciones} finalizada.")

                lista_analisis.append(salida)
                lista_ficheros.append(fichero_ejecucion_output)
            
            salida = self._consolidaAnalisisGeneral(model, fichero_ouput, lista_ficheros, lista_analisis)
            lista_salida.append(salida)

        logging.info(f"{self._clase}.{method}-end Fin de proceso para el modelo {model}")

        logging.info(f"{self._clase}.{method}---------------------------------------------------")
        logging.info(f"{self._clase}.{method} END")
        logging.info(f"{self._clase}.{method}---------------------------------------------------")

        return lista_salida 

    def genera_propuestas(self, modelos:list) -> None:
        """
        Genera propuestas de solución para una incidencia.
        (De momento sin implementar)
        """
        logging.info(
            "Llamada a genera_propuestas(incidencia=%s) (no implementado)",
            incidencia,
        )
        pass

    def genera_prompt_app(self, model:str, modelo_generador_prompt:str="qwen3:30b-a3b-thinking-2507-q4_K_M") -> str:
        """
        Genera el prompt específico para una app dado un contexto de incidencia.
        (De momento sin implementar)
        """
        method="genera_prompt_app"
        logging.info(f"{self._clase}.{method}-----------------------------------------------------")
        logging.info(f"{self._clase}.{method}-START Inicio de causa raíz, clasificación y solución")
        logging.info(f"{self._clase}.{method}- Análisis incidencia a incidencia.")
        logging.info(f"{self._clase}.{method}- modelo que se recupera: {model} , usando modelo generador {modelo_generador_prompt}")
        logging.info(f"{self._clase}.{method}- temperatura: {self._temperature}, aplicación: {self._app_name}")
        logging.info(f"{self._clase}.{method}-----------------------------------------------------")

        #1. Conseguimos prompt para generar el prompt. 
        prompt = self._getPrompt(self._app_name, PROMPT_GENERA_PROMPT)

        #2. Recuperamos la información del fichero para el modelo que se indica. 
        model_name = model.replace(':', '_')
        fichero_input_path=f"{self._fichero_incidencias.replace('.xlsx','')}_{model_name}.xlsx"

        with open(fichero_input_path, "r", encoding="utf-8") as f:
            content = f.read()
        
        analisis_incidencias, cot = ollama_manager.separa_thinking_contenido(content)
        prompt_formateado=prompt.replace("{analisis_incidencias}", analisis_incidencias)

        salida = ollama_manager.call_ollama_api(modelo_generador_prompt, "", prompt_formateado)
        content, thinking = ollama_manager.separa_thinking_contenido(salida)
    
        logging.info(f"{self._clase}.{method}---------------------------------------------------")
        logging.info(f"{self._clase}.{method} END")
        logging.info(f"{self._clase}.{method}---------------------------------------------------")

        return content 
    
    #######
    # Métodos que van a iterar para todas las incidenicas.

    def analiza_incidencia_causa_raiz(self, lista_campos:list, fichero_output=None) -> None:
        """
        The function `analiza_incidencia_causa_raiz` is the third method to use in the workflow of incident analysis. 
        
        It performs for every model given, an llm analisi to generate a specific classification, resolution and root casuse of the incident.

        It saves all the information generated in a file (using original name + sufix with the model name), and also, 
        saves the DataFrame processed in an internal dictionary name. 

        The DataFframe processed is the original + three new fields:
        - classification --> Description resume of the incident.
        - rootcause --> A brief description of the rootcause
        - solution --> Solution made in the ticket.

        Also, are added another three fields:
        - status --> to know if it has benn processed or not, 
        - cot --> The Chain of Thought returned by the model,
        - model --> The model used to generate the information.

        Parameters:
            lista_campos:list --> The name of the fields that we need use from the data frame loaded. First. field must be the ID for logging reasons.

        Exception:
            Raises an IncidenException if validations are not superated. The lista_campos fields has not all elements, it hasn't be assigned the models, etc.

        Return:
            None
        """
        method="analiza_incidencia_causa_raiz"
        logging.info(f"{self._clase}.{method}-----------------------------------------------------")
        logging.info(f"{self._clase}.{method}-START Inicio de causa raíz, clasificación y solución")
        logging.info(f"{self._clase}.{method}- Análisis incidencia a incidencia.")
        logging.info(f"{self._clase}.{method}- modelos a usar: {self._modelos}, razonamiento para modelos generales: {self._thinking}")
        logging.info(f"{self._clase}.{method}- temperatura: {self._temperature}, aplicación: {self._app_name}")
        logging.info(f"{self._clase}.{method}-----------------------------------------------------")

        #Recuperamos el promtp. Si no se encuentra en la lista de prompts, calculamos uno y lo guardamos en fichero.
        try:
            existe_prompt = True 
            prompt = self._getPrompt(self._app_name, PROMPT_ANALIZA_CAUSA_RAIZ)
            logging.debug(f"{self._clase}.{method}- Prompt a usar existente: {prompt}")    
        except Exception as e:
            #Si salta excepción es que no tenemos prompt y lo tenemos que calcular.
            existe_prompt = False

        sufijo = "cr"

        #Validamos que recibimos todos los campos y preparamos la lista de campos que están en el prompt
        #Estas variables se tienen que generar en tiempo de ejecución para cada incidencia, por eso se tienen que pasar al método _procesaIncidencia.
        if len(lista_campos) != 3 :
            raise IncidenciaException(f"{method}-ERROR La lista de campos no tiene los 2 elementos necesarios: desc, res.")

        lista_variables = ["desc", "res", "ticket_type"]
    
        #Iteramos para cada modelo
        for model in self._modelos:
            logging.info(f"{self._clase}.{method}-start Inicio de proceso para el modelo {model}")
            model_name = model.replace(':', '_')

            #Si no existe prompt lo generamos y lo guardamos en fichero.
            if existe_prompt == False:
                logging.info(f"{self._clase}.{method}- calculando prompt porque no existe analisis causa raíz para la app {self._app_name}")
                prompt = self.genera_prompt_app(model)

                fichero_output = f"{self._fichero_incidencias.replace('.xlsx','')}_prompt_CR_{model_name}.txt"
                with open(fichero_output, "w") as f:
                    f.write(prompt)
            else:
                logging.info(f"{self._clase}.{method}- Prompt recuperado")

            #Copiamos el DF.
            df_proceso = self._getDf(model_name)
            df_nuevo = None
            if df_proceso is None:
                logging.debug(f"{method}- Columnas recuperadas: {df_proceso.columns}")
                df_nuevo = df_proceso.copy()
                df_nuevo[f'status_{sufijo}'] = "PEND"
                df_nuevo[f'cot_{sufijo}'] = "'"
                df_nuevo[f'model{sufijo}'] = model
                logging.debug(f"{method}- Columnas actualizadas: {df_nuevo.columns}")

                #Grabamos el Excel para meter las columnas nuevas.
                fichero_ouput_path=f"{self._fichero_incidencias.replace('.xlsx','')}_{model_name}.xlsx"
                df_nuevo.to_excel(fichero_ouput_path, index=False)
            else:
                logging.error(f"{method}-ERROR no hay df_proceso")
            
            #Procesamos el DF completo, sabiendo que es posible que esté realizado.
            self._procesaIncidencias(df_nuevo, sufijo, model, prompt, lista_variables, lista_campos)
            logging.info(f"{self._clase}.{method}-end Fin de proceso para el modelo {model}")

        
        logging.info(f"{self._clase}.{method}---------------------------------------------------")
        logging.info(f"{self._clase}.{method} END")
        logging.info(f"{self._clase}.{method}---------------------------------------------------")

    def calcula_descripcion_solucion(self, lista_campos:list, lista_variables = None) -> None:
        """
        The function `calcula_descripcion_solucion` is the first method to use in the workflow of incident analysis. 
        
        It performs for every model given, an llm analisi to generate a specific resume of the incident description and the 
        solution description. 

        It saves all the information generated in a file (using original name + sufix with the model name), and also, 
        saves the DataFrame processed in an internal dictionary name. 

        The DataFframe processed is the original + three new fields:
        - desc --> Description resume of the incident.
        - res --> Incident Solution resumen. 
        - res_type --> Type of resolution generated: real (means the incident has a real resolution included), estimated (means the incident doesn't have included a real resolution)

        Also, are added another three fields:
        - status --> to know if it has benn processed or not, 
        - cot --> The Chain of Thought returned by the model,
        - model --> The model used to generate the information.

        Parameters:
            lista_campos:list --> The name of the fields that we need use from the data frame loaded. First. field must be the ID for logging reasons.

        Exception:
            Raises an IncidenException if validations are not superated. The lista_campos fields has not all elements, it hasn't be assigned the models, etc.

        Return:
            None
        """
        method="calcula_descripcion_solucion"
        logging.info(f"{self._clase}.{method}---------------------------------------------")
        logging.info(f"{self._clase}.{method}-START Inicio cálculo Descripción y Solución.")
        logging.info(f"{self._clase}.{method}- Análisis incidencia a incidencia.")
        logging.info(f"{self._clase}.{method}- modelos a usar: {self._modelos}, razonamiento para modelos generales: {self._thinking}")
        logging.info(f"{self._clase}.{method}- temperatura: {self._temperature}, aplicación: {self._app_name}")
        logging.info(f"{self._clase}.{method}---------------------------------------------")

        prompt = self._getPrompt(self._app_name, PROMPT_GENERA_DESC_SOL)
        logging.debug(f"{self._clase}.{method}- Prompt a usar: ")

        sufijo = "cds"

        ### 
        #Validamos que recibimos todos los campos y preparamos la lista de campos que están en el prompt
        #Estas variables se tienen que generar en tiempo de ejecución para cada incidencia, por eso se tienen que pasar al método _procesaIncidencia.
        if len(lista_campos) < 6 :
            raise IncidenciaException(f"{method}-ERROR La lista de campos no tiene los 6 elementos necesarios: id, descripcion, descripcion corta, nota resolución, nota de trabajo, estado.")

        if self._modelos is None:
            raise IncidenciaException(f"{method}-ERROR No se han especificado los modelos a usar.")
        ### 
        if lista_variables is None:
            lista_variables = ["id", "description", "descripcion_corta", "nota_resolucion", "nota_trabajo", "comentario_nota_trabajo", "estado", "app_name"]
    
        #Iteramos para cada modelo
        for model in self._modelos:
            logging.info(f"{self._clase}.{method}-start Inicio de proceso para el modelo {model}")

            model_name = model.replace(':', '_')
            
            #Copiamos el DF.
            df_proceso = self._getDf(model_name)
            if df_proceso is None:
                df_proceso = self._df.copy()
                df_proceso[f'status_{sufijo}'] = "PEND"
                df_proceso[f'cot_{sufijo}'] = ""
                df_proceso[f'model{sufijo}'] = model
            
            #Procesamos el DF completo, sabiendo que es posible que esté realizado.
            df_local = self._procesaIncidencias(df_proceso, sufijo, model, prompt, lista_variables, lista_campos)
            self._setDf(model, df_local)
            logging.info(f"{self._clase}.{method}-end Fin de proceso para el modelo {model}")

        logging.info(f"{self._clase}.{method}---------------------------------------------------")
        logging.info(f"{self._clase}.{method} END")
        logging.info(f"{self._clase}.{method}---------------------------------------------------")

In [8]:
#####################################################################################
#####################################################################################
#####################################################################################

In [9]:
##ON BOARDING 

import logging
import time 

root = logging.getLogger()
for h in root.handlers[:]:
    root.removeHandler(h)
    h.close()

logging.basicConfig(level=logging.INFO,format="%(asctime)s %(levelname)s %(name)s - %(message)s")

class HttpRequestFilter(logging.Filter):
    def filter(self, record):
        # Comprueba si el mensaje contiene la cadena específica del HTTP request
        return "HTTP Request" not in record.getMessage()

for handler in logging.getLogger().handlers:
    handler.addFilter(HttpRequestFilter())

#Variables ruta
usuario = 'Users/zzddfge'
carpeta = 'trabajo/incidencias'
disco_duro = '/Volumes/Macintosh HD'

url_mac=os.path.join(disco_duro, usuario, carpeta)

#Variables modelos
modelo_ministral="ministral-3:14b"
modelo_nemotron="nemotron-cascade-2:latest"
modelo_qwen_mlx="qwen3.6:35b-a3b-nvfp4"
modelo_glm="glm-4.7-flash:latest"
modelo_openia="gpt-oss:latest"
modelo_gemma="gemma4:26b-mlx"

#Varaibles App
fichero_onboarding="incidencias_onboarding.xlsx"
app="OnBoarding"

#Creación de la clase 
fichero = fichero_onboarding
modelo=modelo_gemma

incidencias = Incidencias(app, os.path.join(url_mac, fichero))
logging.info(incidencias._app_name)

#Paso 1 - Tratamos datos de descripción y solución. 
incidencias.setDatosModelos([modelo], 0.7, True)
lista_campos_excel=["número", "descripción", "descripción corta", "notas de resolución", "notas de trabajo", "comentarios y notas de trabajo", "estado del incidente","servicio"]


#Tiempo por defecto de timeout = 300 segundos, 5 min. Si una incidencia tarda más de 5 minutos se vuelve a ejecutar.
incidencias.calcula_descripcion_solucion(lista_campos_excel)


#Paso 2 - Tratamos análisis general, pero con otro modelo. 
incidencias.setModeloEntreFases(modelo)
incidencias.setDatosModelos([modelo], 0.7, True)

#Aumentamos el timeout a 1 hora.
incidencias.setTimeout(7200.0)
lista_analisis = incidencias.analiza_todas_incidencias(context_size=32000)





2026-05-23 15:47:04,028 INFO root - Incidencias._loadIncidencias-START - Leyendo el archivo Excel: /Volumes/Macintosh HD/Users/zzddfge/trabajo/incidencias/incidencias_onboarding.xlsx
2026-05-23 15:47:04,298 INFO root - Incidencias._loadIncidencias- 316 registros leídos.
2026-05-23 15:47:04,300 INFO root - OnBoarding
2026-05-23 15:47:04,300 INFO root - Incidencias.calcula_descripcion_solucion---------------------------------------------
2026-05-23 15:47:04,300 INFO root - Incidencias.calcula_descripcion_solucion-START Inicio cálculo Descripción y Solución.
2026-05-23 15:47:04,300 INFO root - Incidencias.calcula_descripcion_solucion- Análisis incidencia a incidencia.
2026-05-23 15:47:04,301 INFO root - Incidencias.calcula_descripcion_solucion- modelos a usar: ['gemma4:26b-mlx'], razonamiento para modelos generales: True
2026-05-23 15:47:04,301 INFO root - Incidencias.calcula_descripcion_solucion- temperatura: 0.7, aplicación: OnBoarding
2026-05-23 15:47:04,301 INFO root - Incidencias.cal

KeyboardInterrupt: 

In [29]:

fichero_onboarding="incidencias_onboarding.xlsx"

#Paso 3 - Tratamos el análisis general para todas las incidencias. 
incidencias.setTimeout(300.0)
lista_campos_excel=["desc", "res", "ticket_type"]
incidencias.analiza_incidencia_causa_raiz(lista_campos_excel)

2026-04-21 08:26:21,704 INFO root - Incidencias.analiza_incidencia_causa_raiz-----------------------------------------------------
2026-04-21 08:26:21,704 INFO root - Incidencias.analiza_incidencia_causa_raiz-START Inicio de causa raíz, clasificación y solución
2026-04-21 08:26:21,705 INFO root - Incidencias.analiza_incidencia_causa_raiz- Análisis incidencia a incidencia.
2026-04-21 08:26:21,705 INFO root - Incidencias.analiza_incidencia_causa_raiz- modelos a usar: ['qwen3.6:35b-a3b-nvfp4'], razonamiento para modelos generales: True
2026-04-21 08:26:21,706 INFO root - Incidencias.analiza_incidencia_causa_raiz- temperatura: 0.7, aplicación: OnBoarding
2026-04-21 08:26:21,706 INFO root - Incidencias.analiza_incidencia_causa_raiz-----------------------------------------------------
2026-04-21 08:26:21,706 INFO root - Incidencias.analiza_incidencia_causa_raiz-start Inicio de proceso para el modelo qwen3.6:35b-a3b-nvfp4
2026-04-21 08:26:21,707 INFO root - Incidencias.analiza_incidencia_caus

In [1]:
prompt_en_en = """ 
#ROLE
You are a top-class incident analyst, focused on reducing the number of tickets.
You have been hired to normalize an incident description and the incident resolution
of a given ticket.
The name of the application is `on boarding`

#INPUT DATA
The following fields are used to calculate the outputs.
* id: ticket code.
* description: problem description. Sometimes blank as it is autogenerated.
* description_short: short description.
* resolution_note: resolution notes. Sometimes blank as the ticket is not closed yet.
* work_note: working notes.
* work_note_comment: more working notes (optional).
* status: status of the ticket.

#TASK
You must produce FIVE fields:
1) desc: short description of the problem/request (only what happens is needed).
2) res: short description of the action/solution.
3) rootcause: canonical root cause label (max 10 words).
4) res_type: Real | Estimado
5) ticket_type: Incidencia | Peticion | Indeterminado

#FIELD RULES (STRICT)
- Language: All JSON string values must be in English. All your chain of thoughts must be English.
- Create five new fields: desc, res, rootcause, res_type, ticket_type.
- Output JSON must include the 5 fields in the exact order:
  {{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}

#INTERNAL ANALYSIS STEPS (MANDATORY)
Before generating the final JSON, do these steps internally (do NOT output them):
A) Extract:
   A1) OBBSERVED FACTS: what is wrong / what fails / what impact is there?
   A2) REQUESTED ACTION: What is the user or the operator asking for?
B) Decide ticket_type using STRICT rules (see DISAMBIGUATION).
C) Generate desc/res/rootcause/res_type consistent with ticket_type.

#DISAMBIGUATION (STRICT)  <-- THIS IS CRITICAL
You MUST NOT classify based on keywords alone (e.g., "modificar", "actualizar", "cambiar").

## Assign ticket_type = Incidencia when ANY of these is true:
- There are signals of malfunction/defect/impact, e.g.:
  "wrong", "incorrect", "incomplete", "don't correspond", "descuadre",
  "save didn't work", "can't update", "error", "fail", "don't work", "blocks",
  "inconsistency", "impact", "affect", "don't let", "don't load", "don't validate".
- The request is to modify/update data BECAUSE the data is wrong or the system registered it incorrectly
  (e.g., "solve", "modify", "subsanar", "revert", "data bad informed").
- There is mention of a technical/process defect cause (bug, integración, migración, carga, parametrización).

## Assign ticket_type = Peticion when ALL of these are true:
- No signals of malfunction/defect/impact (no "bad/incorrect/error/fail/block" etc.).
- The text is a service request or consultation, such as:
  - A status check (“status?”, “follow-up”, “when will it be done?”, “information”).
  - A voluntary request for change/onboarding/offboarding/configuration/access/permissions WITHOUT indicating something is wrong.

## Assign ticket_type = Indeterminado when:
- There is insufficient information and you cannot infer whether there is a defect/impact or a voluntary request.

#DESC FIELD
- Create *desc* as a short and concise summary of what happens or is needed.
- desc must NOT include solution nor root cause, only the situation.
- If ticket_type = Request, desc should reflect the request/consultation.
- If ticket_type = Incident, desc should reflect the problem/impact.

#RES FIELD
- Create *res* with a short description of the solution/action taken.
- If ticket_type = Request:
  - If it is only a consultation (status/info) and no manual action is required:
    res = "This ticket is a request, not an incident."
  - If it is a request that requires a manual action (e.g., add/remove/change of permissions/configuration):
    res = "This ticket is a request, not an incident. The request action was done."
    (If no evidence of completion exists, omit the second sentence.)
- If ticket_type = Incidencia:
  - res must describe the real practical action taken to resolve it, if present.
  - If no clear solution is contained but it is clear that a manual action was done by the operator:
    res = "This incident doesn't have a clear solution, but it has been done a manual action by operator."
  - Otherwise, if there is no explicit resolution:
    res = "This incident doesn't have a clear solution."

#ROOTCAUSE FIELD
- Create *rootcause* with max 10 words.
- Use canonical labels: normalize equivalent meanings to the same short label.
- Do NOT invent causes.

## Rootcause rules by ticket_type:
- If ticket_type = Request:
  - If it is a status/info request: rootcause = "Status information request."
  - If it is a service request (add/remove/change/access): rootcause = "Service request correctly."
- If ticket_type = Incident:
  - If the ticket is closed as duplicated: rootcause = "Dupplicated incident."
  - If you cannot understand rootcause but manual action is clear:
    rootcause = "No clear rootcause; manual action let go on."
  - If you cannot understand rootcause:
    rootcause = "No clear rootcause"

#RES_TYPE FIELD
- res_type can only be: Real, Stimated
- Use Real when:
  - ticket_type = Incident AND there is a clear action/solution done.
  - OR ticket_type = Request AND there is evidence that the requested management was performed.
- Use Stimated in all other cases.

#FEW-SHOT EXAMPLES (IMPORTANT)

## Example 1: INCIDENT (data modification by error)
Input:
"Customer CIF modification: bad informated and bill affectation."
Output:
{{"desc":"Bad customer CIF and bill affectation.","res":"CIF updated in customer master.","rootcause":"Incorrect maser data.","res_type":"Real","ticket_type":"Incident"}}

##Example 2: REQUEST (voluntary change without failure)
Input:
“I request changing the customer’s CIF due to a corporate change. There are no errors.”
Output:
{{“desc”:“Request to change the CIF due to a corporate change.”,“res”:“This ticket is a request, not an incident.”,“rootcause”:“Service request without failure.”,“res_type”:“Estimated”,“ticket_type”:“Request”}}

##Example 3: INCIDENT (functional failure)
Input:
“The system does not update the address even though I save it.”
Output:
{{“desc”:“The address is not updated when saved.”,“res”:“A fix was applied to allow the address to be updated.”,“rootcause”:“Failure in data update.”,“res_type”:“Actual”,“ticket_type”:“Incident”}}

##Example 4: REQUEST (status inquiry)
Input:
“What is the status of ticket 123? When will it be resolved?”
Output:
{{“desc”:“Status inquiry for ticket 123.”,“res”:“This ticket is a request, not an incident.”,“rootcause”:“Status or information inquiry.”,“res_type”:“Estimated”,“ticket_type”:“Request”}}

##Example 5: INCIDENT (integration)
Input:
“The IBAN was loaded incorrectly by the integration.”
Output:
{{“desc”:“IBAN loaded incorrectly by the integration.”,“res”:“The IBAN was corrected and the integration was reviewed.”,“rootcause”:“Integration error during data loading.”,“res_type”:“Actual”,“ticket_type”:“Incident”}}

##Example 6: DUPLICATE
Input:
“Close, duplicate of INC-999.”
Output:
{{“desc”:“Request to close due to duplicate incident.”,“res”:“The ticket was closed as a duplicate.”,“rootcause”:“Duplicate incident.”,“res_type”:“Actual”,“ticket_type”:“Incident”}}

#QUALITY CHECK (before finalizing)
- Ensure all new fields are in English.
- Ensure the 5 fields are present and in the exact order.
- Ensure rootcause has max 10 words.
- Ensure ticket_type matches DISAMBIGUATION rules.

#INPUT
<id>INC000202519091</id>
<description>The onboarding HRSC00000968494 was mistakenly cancelled by Enek Service Desk, but it needs to be reactivated. The onboardee starts on 12/31/2025, and there are still pending tasks to be completed, including the preparation of the contract. I am attaching emails regarding this matter.

Contact People: ES22742787L
Name: BEGOÑA REDONDO ABASCAL
Email: BEGONA.REDONDO@ENEL.COM
Location: Spain-ZARAGOZA-CL AZNAR MOLINA 2_8
Business phone:
Alternative Telephone: +34 625 60 63 74
Time Slot: Anytime
Alternative Email: BEGONA.REDONDO@ENEL.COM</description>
<description_short>The onboarding HRSC00000968494 was mistakenly cancelled by Enek Service Desk, but it must be reactivated.</description_short>
<resolution_note>12/30/2025 10:10:38 - Hugo Emanuel Jesus Militao Dias da Silva (Work notes)
The user calls to get information and is advised to reopen the Onboarding ticket in order to regularize the user’s active status.</resolution_note>
<work_note>Dear user,
unfortunately, the deletion of the Onboarding case has caused some inconsistencies/issues in its management.

Regarding the Assets, Infrastructure, Workspace, and Badge RITMs, these were moved to Closed Incomplete as a result of the deletion. Unfortunately, it is not possible to trigger new ones; therefore, we recommend contacting the Asset Management Team for further support.

The "Asignación email y Activación Internet y VPN" task is in Closed Complete status. If the Onboardee is still experiencing VPN issues, we recommend opening a new Incident with the competent support team (which is not our team).

Thank you and regards,
Sara Cicia</work_note>
<work_note_comment>05/01/2026 11:49:04 - Sara Cicia (Additional comments)
Resolution Notes: Dear user,
unfortunately, the deletion of the Onboarding case has caused some inconsistencies/issues in its management.

Regarding the Assets, Infrastructure, Workspace, and Badge RITMs, these were moved to Closed Incomplete as a result of the deletion. Unfortunately, it is not possible to trigger new ones; therefore, we recommend contacting the Asset Management Team for further support.

The “Email assignment and Internet and VPN activation” task is in Closed Complete status. If the Onboardee is still experiencing VPN issues, we recommend opening a new Incident with the competent support team (which is not our team).

Thank you and regards,
Sara Cicia

02/01/2026 17:10:25 - Martina Nachira (Additional comments)
Dear user,

we are verifying the issue.

Regards

31/12/2025 09:24:44 - BEGOÑA REDONDO ABASCAL (Additional comments)
In the screenshot Captura de pantalla 2025-12-31 090208, it can be seen that the Email assignment and Internet and VPN activation task is still in “Ready” status, and in the screenshot IMG-20251231-WA0000 (002) that I have just attached, the error shown to the onboardee can be seen.

31/12/2025 09:15:49 - BEGOÑA REDONDO ABASCAL (Additional comments)
Good morning, Pablo has joined today, but regarding email, mobile phone delivery, access card, etc., the onboarding has not worked. He has not received either of them. And regarding his user account, although an email has been assigned to him: pablo.fernandezgarcia@enel.com, it is giving him an error, showing a message that it is blocked, and he also does not have access to Global Protect. I do not know if this could be related to the fact that the tasks corresponding to physical access card, infrastructure, workstation setup, and smartphone were left in “Closed Incomplete” status (I attach a screenshot of their current status).

30/12/2025 11:21:08 - BEGOÑA REDONDO ABASCAL (Additional comments)
The “Day 1 Presence Confirmation” task should be progressed by the responsible person when the onboarding day arrives, which is tomorrow. I am mainly asking about the task related to mobile phone delivery. If he does not receive the mobile phone, he cannot access the systems, even if his user account has been created.

30/12/2025 11:00:11 - Martina Nachira (Additional comments)
Dear user,

the creation/configuration of emails, Teams, and VPNs through the Onboarding process occurs when the “Onboardee Presence Confirmation Day 1” task is closed.

For PABLO FERNANDEZ GARCIA, the task HRST00001829450, assigned to FERNANDO JOAQUIN MELERO RIVAS, is still in Ready status; therefore, it is necessary to manage the ticket correctly.

Thank you and regards,
Martina Nachira

30/12/2025 10:15:54 - BEGOÑA REDONDO ABASCAL (Additional comments)
The user - Pablo Fernández García - has already been created, but the tasks related to smartphone, physical access card, workstation setup, infrastructure, etc., have been closed as incomplete. See the screenshot added this morning. Pablo joins tomorrow and will need his phone to be able to access the systems.

30/12/2025 10:10:38 - Hugo Emanuel Jesus Militao Dias da Silva (Work notes)
The user calls to request information and is advised to reopen the Onboarding incident in order to regularize the user’s active status.

26/12/2025 08:23:35 - BEGOÑA REDONDO ABASCAL (Additional comments)
I attach a screenshot of tasks where the status appears as “Closed Incomplete”. I am not sure if this may interfere with their normal progression, for example, the delivery of the mobile phone and the access card.

24/12/2025 09:07:39 - Sara Cicia (Additional comments)
Resolution Notes: Dear user,
I have restored the case; however, the associated workflow was automatically deleted. As a result, the tasks may not be triggered automatically in accordance with the standard process. Please monitor the case and inform us should it be necessary to manually unlock any tasks.

In particular, please pay attention to the “Onboardee Presence Confirmation – Day 1” task, as its closure triggers the second integration with CompAC for user update and email configuration.

Thank you and regards,
Sara Cicia
</work_note_comment>
<estado>CLOSED</estado>

#FINAL OUTPUT FORMAT (return only this JSON)
{{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}
"""

prompt_chino_en = """ 
#ROLE
You are a top-class incident analyst, focused on reducing the number of tickets.
You have been hired to normalize an incident description and the incident resolution
of a given ticket.
The name of the application is `on boarding`

#INPUT DATA
The following fields are used to calculate the outputs.
* id: ticket code.
* description: problem description. Sometimes blank as it is autogenerated.
* description_short: short description.
* resolution_note: resolution notes. Sometimes blank as the ticket is not closed yet.
* work_note: working notes.
* work_note_comment: more working notes (optional).
* status: status of the ticket.

#TASK
You must produce FIVE fields:
1) desc: short description of the problem/request (only what happens is needed).
2) res: short description of the action/solution.
3) rootcause: canonical root cause label (max 10 words).
4) res_type: Real | Estimado
5) ticket_type: Incidencia | Peticion | Indeterminado

#FIELD RULES (STRICT)
- Language: All JSON string values must be in English. All your chain of thoughts must be Chinesse.
- Create five new fields: desc, res, rootcause, res_type, ticket_type.
- Output JSON must include the 5 fields in the exact order:
  {{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}

#INTERNAL ANALYSIS STEPS (MANDATORY)
Before generating the final JSON, do these steps internally (do NOT output them):
A) Extract:
   A1) OBBSERVED FACTS: what is wrong / what fails / what impact is there?
   A2) REQUESTED ACTION: What is the user or the operator asking for?
B) Decide ticket_type using STRICT rules (see DISAMBIGUATION).
C) Generate desc/res/rootcause/res_type consistent with ticket_type.

#DISAMBIGUATION (STRICT)  <-- THIS IS CRITICAL
You MUST NOT classify based on keywords alone (e.g., "modificar", "actualizar", "cambiar").

## Assign ticket_type = Incidencia when ANY of these is true:
- There are signals of malfunction/defect/impact, e.g.:
  "wrong", "incorrect", "incomplete", "don't correspond", "descuadre",
  "save didn't work", "can't update", "error", "fail", "don't work", "blocks",
  "inconsistency", "impact", "affect", "don't let", "don't load", "don't validate".
- The request is to modify/update data BECAUSE the data is wrong or the system registered it incorrectly
  (e.g., "solve", "modify", "subsanar", "revert", "data bad informed").
- There is mention of a technical/process defect cause (bug, integración, migración, carga, parametrización).

## Assign ticket_type = Peticion when ALL of these are true:
- No signals of malfunction/defect/impact (no "bad/incorrect/error/fail/block" etc.).
- The text is a service request or consultation, such as:
  - A status check (“status?”, “follow-up”, “when will it be done?”, “information”).
  - A voluntary request for change/onboarding/offboarding/configuration/access/permissions WITHOUT indicating something is wrong.

## Assign ticket_type = Indeterminado when:
- There is insufficient information and you cannot infer whether there is a defect/impact or a voluntary request.

#DESC FIELD
- Create *desc* as a short and concise summary of what happens or is needed.
- desc must NOT include solution nor root cause, only the situation.
- If ticket_type = Request, desc should reflect the request/consultation.
- If ticket_type = Incident, desc should reflect the problem/impact.

#RES FIELD
- Create *res* with a short description of the solution/action taken.
- If ticket_type = Request:
  - If it is only a consultation (status/info) and no manual action is required:
    res = "This ticket is a request, not an incident."
  - If it is a request that requires a manual action (e.g., add/remove/change of permissions/configuration):
    res = "This ticket is a request, not an incident. The request action was done."
    (If no evidence of completion exists, omit the second sentence.)
- If ticket_type = Incidencia:
  - res must describe the real practical action taken to resolve it, if present.
  - If no clear solution is contained but it is clear that a manual action was done by the operator:
    res = "This incident doesn't have a clear solution, but it has been done a manual action by operator."
  - Otherwise, if there is no explicit resolution:
    res = "This incident doesn't have a clear solution."

#ROOTCAUSE FIELD
- Create *rootcause* with max 10 words.
- Use canonical labels: normalize equivalent meanings to the same short label.
- Do NOT invent causes.

## Rootcause rules by ticket_type:
- If ticket_type = Request:
  - If it is a status/info request: rootcause = "Status information request."
  - If it is a service request (add/remove/change/access): rootcause = "Service request correctly."
- If ticket_type = Incident:
  - If the ticket is closed as duplicated: rootcause = "Dupplicated incident."
  - If you cannot understand rootcause but manual action is clear:
    rootcause = "No clear rootcause; manual action let go on."
  - If you cannot understand rootcause:
    rootcause = "No clear rootcause"

#RES_TYPE FIELD
- res_type can only be: Real, Stimated
- Use Real when:
  - ticket_type = Incident AND there is a clear action/solution done.
  - OR ticket_type = Request AND there is evidence that the requested management was performed.
- Use Stimated in all other cases.

#FEW-SHOT EXAMPLES (IMPORTANT)

## Example 1: INCIDENT (data modification by error)
Input:
"Customer CIF modification: bad informated and bill affectation."
Output:
{{"desc":"Bad customer CIF and bill affectation.","res":"CIF updated in customer master.","rootcause":"Incorrect maser data.","res_type":"Real","ticket_type":"Incident"}}

##Example 2: REQUEST (voluntary change without failure)
Input:
“I request changing the customer’s CIF due to a corporate change. There are no errors.”
Output:
{{“desc”:“Request to change the CIF due to a corporate change.”,“res”:“This ticket is a request, not an incident.”,“rootcause”:“Service request without failure.”,“res_type”:“Estimated”,“ticket_type”:“Request”}}

##Example 3: INCIDENT (functional failure)
Input:
“The system does not update the address even though I save it.”
Output:
{{“desc”:“The address is not updated when saved.”,“res”:“A fix was applied to allow the address to be updated.”,“rootcause”:“Failure in data update.”,“res_type”:“Actual”,“ticket_type”:“Incident”}}

##Example 4: REQUEST (status inquiry)
Input:
“What is the status of ticket 123? When will it be resolved?”
Output:
{{“desc”:“Status inquiry for ticket 123.”,“res”:“This ticket is a request, not an incident.”,“rootcause”:“Status or information inquiry.”,“res_type”:“Estimated”,“ticket_type”:“Request”}}

##Example 5: INCIDENT (integration)
Input:
“The IBAN was loaded incorrectly by the integration.”
Output:
{{“desc”:“IBAN loaded incorrectly by the integration.”,“res”:“The IBAN was corrected and the integration was reviewed.”,“rootcause”:“Integration error during data loading.”,“res_type”:“Actual”,“ticket_type”:“Incident”}}

##Example 6: DUPLICATE
Input:
“Close, duplicate of INC-999.”
Output:
{{“desc”:“Request to close due to duplicate incident.”,“res”:“The ticket was closed as a duplicate.”,“rootcause”:“Duplicate incident.”,“res_type”:“Actual”,“ticket_type”:“Incident”}}

#QUALITY CHECK (before finalizing)
- Ensure all new fields are in English.
- Ensure the 5 fields are present and in the exact order.
- Ensure rootcause has max 10 words.
- Ensure ticket_type matches DISAMBIGUATION rules.

#INPUT
<id>INC000202519091</id>
<description>The onboarding HRSC00000968494 was mistakenly cancelled by Enek Service Desk, but it needs to be reactivated. The onboardee starts on 12/31/2025, and there are still pending tasks to be completed, including the preparation of the contract. I am attaching emails regarding this matter.

Contact People: ES22742787L
Name: BEGOÑA REDONDO ABASCAL
Email: BEGONA.REDONDO@ENEL.COM
Location: Spain-ZARAGOZA-CL AZNAR MOLINA 2_8
Business phone:
Alternative Telephone: +34 625 60 63 74
Time Slot: Anytime
Alternative Email: BEGONA.REDONDO@ENEL.COM</description>
<description_short>The onboarding HRSC00000968494 was mistakenly cancelled by Enek Service Desk, but it must be reactivated.</description_short>
<resolution_note>12/30/2025 10:10:38 - Hugo Emanuel Jesus Militao Dias da Silva (Work notes)
The user calls to get information and is advised to reopen the Onboarding ticket in order to regularize the user’s active status.</resolution_note>
<work_note>Dear user,
unfortunately, the deletion of the Onboarding case has caused some inconsistencies/issues in its management.

Regarding the Assets, Infrastructure, Workspace, and Badge RITMs, these were moved to Closed Incomplete as a result of the deletion. Unfortunately, it is not possible to trigger new ones; therefore, we recommend contacting the Asset Management Team for further support.

The "Asignación email y Activación Internet y VPN" task is in Closed Complete status. If the Onboardee is still experiencing VPN issues, we recommend opening a new Incident with the competent support team (which is not our team).

Thank you and regards,
Sara Cicia</work_note>
<work_note_comment>05/01/2026 11:49:04 - Sara Cicia (Additional comments)
Resolution Notes: Dear user,
unfortunately, the deletion of the Onboarding case has caused some inconsistencies/issues in its management.

Regarding the Assets, Infrastructure, Workspace, and Badge RITMs, these were moved to Closed Incomplete as a result of the deletion. Unfortunately, it is not possible to trigger new ones; therefore, we recommend contacting the Asset Management Team for further support.

The “Email assignment and Internet and VPN activation” task is in Closed Complete status. If the Onboardee is still experiencing VPN issues, we recommend opening a new Incident with the competent support team (which is not our team).

Thank you and regards,
Sara Cicia

02/01/2026 17:10:25 - Martina Nachira (Additional comments)
Dear user,

we are verifying the issue.

Regards

31/12/2025 09:24:44 - BEGOÑA REDONDO ABASCAL (Additional comments)
In the screenshot Captura de pantalla 2025-12-31 090208, it can be seen that the Email assignment and Internet and VPN activation task is still in “Ready” status, and in the screenshot IMG-20251231-WA0000 (002) that I have just attached, the error shown to the onboardee can be seen.

31/12/2025 09:15:49 - BEGOÑA REDONDO ABASCAL (Additional comments)
Good morning, Pablo has joined today, but regarding email, mobile phone delivery, access card, etc., the onboarding has not worked. He has not received either of them. And regarding his user account, although an email has been assigned to him: pablo.fernandezgarcia@enel.com, it is giving him an error, showing a message that it is blocked, and he also does not have access to Global Protect. I do not know if this could be related to the fact that the tasks corresponding to physical access card, infrastructure, workstation setup, and smartphone were left in “Closed Incomplete” status (I attach a screenshot of their current status).

30/12/2025 11:21:08 - BEGOÑA REDONDO ABASCAL (Additional comments)
The “Day 1 Presence Confirmation” task should be progressed by the responsible person when the onboarding day arrives, which is tomorrow. I am mainly asking about the task related to mobile phone delivery. If he does not receive the mobile phone, he cannot access the systems, even if his user account has been created.

30/12/2025 11:00:11 - Martina Nachira (Additional comments)
Dear user,

the creation/configuration of emails, Teams, and VPNs through the Onboarding process occurs when the “Onboardee Presence Confirmation Day 1” task is closed.

For PABLO FERNANDEZ GARCIA, the task HRST00001829450, assigned to FERNANDO JOAQUIN MELERO RIVAS, is still in Ready status; therefore, it is necessary to manage the ticket correctly.

Thank you and regards,
Martina Nachira

30/12/2025 10:15:54 - BEGOÑA REDONDO ABASCAL (Additional comments)
The user - Pablo Fernández García - has already been created, but the tasks related to smartphone, physical access card, workstation setup, infrastructure, etc., have been closed as incomplete. See the screenshot added this morning. Pablo joins tomorrow and will need his phone to be able to access the systems.

30/12/2025 10:10:38 - Hugo Emanuel Jesus Militao Dias da Silva (Work notes)
The user calls to request information and is advised to reopen the Onboarding incident in order to regularize the user’s active status.

26/12/2025 08:23:35 - BEGOÑA REDONDO ABASCAL (Additional comments)
I attach a screenshot of tasks where the status appears as “Closed Incomplete”. I am not sure if this may interfere with their normal progression, for example, the delivery of the mobile phone and the access card.

24/12/2025 09:07:39 - Sara Cicia (Additional comments)
Resolution Notes: Dear user,
I have restored the case; however, the associated workflow was automatically deleted. As a result, the tasks may not be triggered automatically in accordance with the standard process. Please monitor the case and inform us should it be necessary to manually unlock any tasks.

In particular, please pay attention to the “Onboardee Presence Confirmation – Day 1” task, as its closure triggers the second integration with CompAC for user update and email configuration.

Thank you and regards,
Sara Cicia
</work_note_comment>
<estado>CLOSED</estado>

#FINAL OUTPUT FORMAT (return only this JSON)
{{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}
"""

prompt_es_es="""
#ROL
Eres un analista de incidentes de primer nivel, enfocado en reducir el número de tickets.
Has sido contratado para normalizar la descripción de un incidente y la resolución del mismo
de un ticket dado.
El nombre de la aplicación es `on boarding`

#DATOS DE ENTRADA
Los siguientes campos se utilizan para calcular las salidas.
* id: código del ticket.
* descripcion: descripción del problema. A veces está vacío ya que es autogenerado.
* descripcion_corta: descripción corta.
* nota_resolucion: notas de resolución. A veces está vacío ya que el ticket aún no está cerrado.
* nota_trabajo: notas de trabajo.
* comentario_nota_trabajo: más notas de trabajo (opcional).
* estado: estado del ticket.

#TAREA
Debes producir CINCO campos:
1) desc: descripción corta del problema/solicitud (solo se necesita lo que ocurre).
2) res: descripción corta de la acción/solución.
3) rootcause: etiqueta canónica de la causa raíz (máx. 10 palabras).
4) res_type: Real | Estimado
5) ticket_type: Incidencia | Petición | Indeterminado

#REGLAS DE LOS CAMPOS (ESTRICTO)
- Idioma: Todos los valores de cadena JSON deben estar en español. Todo tu razonamiento interno debe estar en español.
- Crea cinco nuevos campos: desc, res, rootcause, res_type, ticket_type.
- El JSON de salida debe incluir los 5 campos en el orden exacto:
   {{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}

#PASOS DE ANÁLISIS INTERNO (OBLIGATORIOS)
Antes de generar el JSON final, realiza estos pasos internamente (NO los generes en la salida):
A) Extraer:
   A1) HECHOS OBSERVADOS: qué está mal / qué falla / qué impacto hay?
   A2) ACCIÓN SOLICITADA: Qué está pidiendo el usuario o el operador?
B) Decidir ticket_type utilizando reglas ESTRICTAS (ver DESAMBIGUACIÓN).
C) Generar desc/res/rootcause/res_type coherentes con ticket_type.

#DESAMBIGUACIÓN (ESTRICTO)   <-- ESTO ES CRÍTICO
NO debes clasificar solo basándote en palabras clave (ej.: "modificar", "actualizar", "cambiar").

## Asignar ticket_type = Incidencia cuando CUALQUIERA de estas sea verdadera:
- Hay señales de mal funcionamiento/defecto/impacto, ej.:
   "mal", "incorrecto", "erróneo", "incompleto", "no corresponde", "descuadre",
   "se grabó mal", "no se actualiza", "error", "fallo", "no funciona", "bloquea",
   "inconsistencia", "impacta", "afecta", "no permite", "no carga", "no valida".
- La solicitud es modificar/actualizar datos DEBIDO A QUE los datos están mal o el sistema los registró incorrectamente
   (ej., "corregir", "rectificar", "subsanar", "revertir", "dato mal informado").
- Hay mención de una causa de defecto técnico/procesal (bug, integración, migración, carga, parametrización).

## Asignar ticket_type = Petición cuando TODAS estas sean verdaderas:
- Sin señales de mal funcionamiento/defecto/impacto (sin "mal/incorrecto/error/fallo/bloqueo" etc.).
- El texto es una solicitud de servicio o consulta, como:
   - Una comprobación de estado (“estado?”, “seguimiento”, “cuándo estará listo?”, “información”).
   - Una solicitud voluntaria de cambio/onboarding/offboarding/configuración/acceso/permisos SIN indicar que algo está mal.

## Asignar ticket_type = Indeterminado cuando:
- Hay información insuficiente y no puedes inferir si hay un defecto/impacto o una solicitud voluntaria.

#CAMPO DESC
- Crear *desc* como un resumen corto y conciso de lo que ocurre o se necesita.
- desc NO debe incluir solución ni causa raíz, solo la situación.
- Si ticket_type = Petición, desc debe reflejar la solicitud/consulta.
- Si ticket_type = Incidencia, desc debe reflejar el problema/impacto.

#CAMPO RES
- Crear *res* con una descripción corta de la solución/acción tomada.
- Si ticket_type = Petición:
   - Si es solo una consulta (estado/info) y no se requiere acción manual:
    res = "Este ticket es una petición, no una incidencia."
   - Si es una solicitud que requiere una acción manual (ej., alta/baja/cambio de permisos/configuración):
    res = "Este ticket es una petición, no una incidencia. Se realizó la gestión solicitada."
     (Si no existe evidencia de finalización, omite la segunda oración.)
- Si ticket_type = Incidencia:
   - res debe describir la acción práctica real tomada para resolverlo, si está presente.
   - Si no se contiene una solución clara pero es claro que se realizó una acción manual por el operador:
    res = "Esta incidencia no tiene una solución clara especificada pero se ha realizado una acción manual por el operador."
   - De lo contrario, si no hay resolución explícita:
    res = "Esta incidencia no tiene una solución clara especificada en la información."

#CAMPO ROOTCAUSE
- Crear *rootcause* con máx. 10 palabras.
- Usar etiquetas canónicas: normalizar significados equivalentes a la misma etiqueta corta.
- NO inventar causas.

## Reglas de rootcause por ticket_type:
- Si ticket_type = Petición:
   - Si es una solicitud de estado/info: rootcause = "Consulta de estado o información."
   - Si es una solicitud de servicio (alta/baja/cambio/acceso): rootcause = "Solicitud de servicio sin fallo."
- Si ticket_type = Incidencia:
   - Si el ticket se cierra como duplicado: rootcause = "Incidencia duplicada."
   - Si no puedes entender la rootcause pero la acción manual es clara:
    rootcause = "Sin rootcause clara; acción manual permitió continuar."
   - Si no puedes entender la rootcause:
    rootcause = "Sin rootcause clara definida en la información."

#CAMPO RES_TYPE
- res_type solo puede ser: Real, Estimado
- Usar Real cuando:
   - ticket_type = Incidencia Y hay una acción/solución clara realizada.
   - O ticket_type = Petición Y hay evidencia de que se realizó la gestión solicitada.
- Usar Estimado en todos los demás casos.

#EJEMPLOS DE POCOS CASOS (IMPORTANTE)

## Ejemplo 1: INCIDENCIA (modificar dato por error)
Entrada:
"Modificar CIF del cliente: está mal informado y afecta facturas."
Salida:
{{"desc":"CIF de cliente incorrecto y afecta facturas.","res":"Se corrigió el CIF en el maestro de clientes.","rootcause":"Dato maestro incorrecto.","res_type":"Real","ticket_type":"Incidencia"}}

## Ejemplo 2: PETICIÓN (cambio voluntario sin fallo)
Entrada:
"Solicito cambiar el CIF del cliente por cambio societario. No hay errores."
Salida:
{{"desc":"Solicitud de cambio de CIF por cambio societario.","res":"Este ticket es una petición, no una incidencia.","rootcause":"Solicitud de servicio sin fallo.","res_type":"Estimado","ticket_type":"Petición"}}

## Ejemplo 3: INCIDENCIA (fallo funcional)
Entrada:
"El sistema no actualiza el domicilio aunque lo guardo."
Salida:
{{"desc":"El domicilio no se actualiza al guardarlo.","res":"Se aplicó corrección para permitir la actualización del domicilio.","rootcause":"Fallo en actualización de datos.","res_type":"Real","ticket_type":"Incidencia"}}

## Ejemplo 4: PETICIÓN (consulta de estado)
Entrada:
"¿Estado del ticket 123? ¿Cuándo estará resuelto?"
Salida:
{{"desc":"Consulta del estado del ticket 123.","res":"Este ticket es una petición, no una incidencia.","rootcause":"Consulta de estado o información.","res_type":"Estimado","ticket_type":"Petición"}}

## Ejemplo 5: INCIDENCIA (integración)
Entrada:
"Se cargó mal el IBAN por integración."
Salida:
{{"desc":"IBAN cargado incorrectamente por la integración.","res":"Se corrigió el IBAN y se revisó la integración.","rootcause":"Error de integración en carga de datos.","res_type":"Real","ticket_type":"Incidencia"}}

## Ejemplo 6: DUPLICADA
Entrada:
"Cerrar, duplicada de INC-999."
Salida:
{{"desc":"Solicitud de cierre por incidencia duplicada.","res":"Se cerró el ticket por duplicidad.","rootcause":"Incidencia duplicada.","res_type":"Real","ticket_type":"Incidencia"}}

#VERIFICACIÓN DE CALIDAD (antes de finalizar)
- Asegurar que todos los nuevos campos estén en español.
- Asegurar que los 5 campos estén presentes y en el orden exacto.
- Asegurar que rootcause tenga máx. 10 palabras.
- Asegurar que ticket_type coincida con las reglas de DESAMBIGUACIÓN.

#ENTRADA
<id>INC000202519091</id>
<descripcion>El Onboarding HRSC00000968494 ha sido cancelado por error por Enek Service Desk, pero debe reactivarse. El Onboardee se incorpora el 31/12/2025, y aún quedan tareas sin completar, entre ellas la elaboración del contrato. Adjunto correos al respecto.

 Persona de contacto:ES22742787L
 Nombre:BEGOÑA REDONDO ABASCAL
 Email:BEGONA.REDONDO@ENEL.COM
 Localización:España-ZARAGOZA-CL AZNAR MOLINA 2_8
 Teléfono de trabajo:
 Teléfono alternativo: +34 625 60 63 74
 Ventana de intervención: Anytime
 Email alternativo: BEGONA.REDONDO@ENEL.COM</descripcion>
<descripcion_corta>El Onboarding HRSC00000968494 ha sido cancelado por error por Enek Service Desk, pero debe reactivar</descripcion_corta>
<nota_resolucion>30/12/2025 10:10:38 - Hugo Emanuel Jesus Militao Dias da Silva (Notas de trabajo)
La usuaria llama parea informarse y se indica que reabra la incidencia de Onboarding para regularizar lo activo dl usuario.</nota_resolucion>
<nota_trabajo>Estimado usuario,
lamentablemente, la eliminación del caso de Onboarding ha provocado algunas inconsistencias/problemas en su gestión.

En cuanto a los RITMs de Activos, Infraestructura, Puesto de trabajo y Tarjeta, estos fueron movidos a Closed Incomplete como consecuencia de la eliminación. Lamentablemente, no es posible generar nuevos; por lo tanto, recomendamos contactar con el equipo de Asset Management para obtener soporte adicional.

La tarea “Asignación email y Activación Internet y VPN” está en estado Closed Complete. Si el onboardee sigue experimentando problemas con la VPN, recomendamos abrir una nueva Incidencia con el equipo de soporte competente (que no es nuestro equipo).

Gracias y un saludo,
Sara Cicia</nota_trabajo>
<comentario_nota_trabajo>05/01/2026 11:49:04 - Sara Cicia (Observaciones adicionales)
Notas de resolución: Estimado usuario,
lamentablemente, la eliminación del caso de Onboarding ha provocado algunas inconsistencias/problemas en su gestión.

En cuanto a los RITMs de Activos, Infraestructura, Puesto de trabajo y Tarjeta, estos fueron movidos a Closed Incomplete como consecuencia de la eliminación. Lamentablemente, no es posible generar nuevos; por lo tanto, recomendamos contactar con el equipo de Asset Management para obtener soporte adicional.

La tarea “Asignación email y Activación Internet y VPN” está en estado Closed Complete. Si el onboardee sigue experimentando problemas con la VPN, recomendamos abrir una nueva Incidencia con el equipo de soporte competente (que no es nuestro equipo).

Gracias y un saludo,
Sara Cicia

02/01/2026 17:10:25 - Martina Nachira (Observaciones adicionales)
Estimado usuario,

estamos verificando el problema.

Saludos

31/12/2025 09:24:44 - BEGOÑA REDONDO ABASCAL (Observaciones adicionales)
En la captura Captura de pantalla 2025-12-31 090208, puede verse que la tarea Asignación email y Activación Internet y VPN sigue estando en “listo”, y en la captura IMG-20251231-WA0000 (002) que acabo de adjuntar se ve el error que aparece al onboardee.

31/12/2025 09:15:49 - BEGOÑA REDONDO ABASCAL (Observaciones adicionales)
Buenos días, Pablo se ha incorporado hoy, pero en lo tocante a correo electrónico, entrega de móvil, tarjeta de acceso, etc., el onboarding no ha funcionado. No ha recibido ninguna de las dos cosas. Y en lo que respecta a su usuario, aunque se le ha asignado correo electrónico: pablo.fernandezgarcia@enel.com, le está dando error, le aparece un mensaje de que está bloqueado y tampoco tiene acceso a Global Protect. No sé si puede tener que ver con que las tareas correspondientes a tarjeta de acceso físico, infraestructuras, alta de puesto de trabajo y smartphone quedaron en estado “cerrado incompleto” (adjunto captura de su estado actual).

30/12/2025 11:21:08 - BEGOÑA REDONDO ABASCAL (Observaciones adicionales)
La tarea de confirmación de presencia “Day 1” corresponde avanzarla al responsable cuando llegue el día de la incorporación, que es mañana. Yo pregunto sobre todo por la tarea correspondiente a la entrega de teléfono móvil. Si no recibe el teléfono móvil no puede acceder a los sistemas, aunque tenga el usuario dado de alta.

30/12/2025 11:00:11 - Martina Nachira (Observaciones adicionales)
Estimado usuario,

la creación/configuración de correos electrónicos, Teams y VPN, a través del proceso de Onboarding, se produce cuando se cierra la tarea Confirmación Presencia Onboardee “Day 1”.

Para PABLO FERNÁNDEZ GARCÍA, la tarea HRST00001829450, asignada a FERNANDO JOAQUÍN MELERO RIVAS, sigue en estado Ready, por lo que es necesario gestionar correctamente el ticket.

Gracias y saludos,
Martina Nachira

30/12/2025 10:15:54 - BEGOÑA REDONDO ABASCAL (Observaciones adicionales)
El usuario - Pablo Fernández García - ya está dado de alta, pero las tareas relativas al smartphone, tarjeta de acceso físico, alta de puesto de trabajo, infraestructuras, etc., se han cerrado como incompletas. Ver captura de pantalla añadida esta mañana. Pablo se incorpora mañana y necesitará su teléfono para poder acceder a los sistemas.

30/12/2025 10:10:38 - Hugo Emanuel Jesus Militao Dias da Silva (Notas de trabajo)
La usuaria llama para informarse y se le indica que reabra la incidencia de Onboarding para regularizar el estado activo del usuario.

26/12/2025 08:23:35 - BEGOÑA REDONDO ABASCAL (Observaciones adicionales)
Adjunto captura de tareas en las que el estado aparece como “closed incomplete”. No sé si esto puede interferir en que continúen su curso normal, por ejemplo, la entrega del teléfono móvil y la tarjeta de acceso.

24/12/2025 09:07:39 - Sara Cicia (Observaciones adicionales)
Notas de resolución: Estimado usuario,
he restaurado el caso; sin embargo, el flujo de trabajo asociado fue eliminado automáticamente. Como resultado, es posible que las tareas no se activen automáticamente conforme al proceso estándar. Por favor, supervise el caso y comuníquenos si es necesario desbloquear manualmente alguna tarea.

En particular, preste atención a la tarea “Confirmación Presencia Onboardee – Day 1”, ya que su cierre activa la segunda integración con CompAC para la actualización del usuario y la configuración del correo electrónico.

Gracias y un saludo,
Sara Cicia
</comentario_nota_trabajo>
<estado>CERRADA</estado>

#FORMATO DE SALIDA FINAL (devolver solo este JSON)
{{"desc": "...", "res": "...", "rootcause":"...", "res_type": "...", "ticket_type": "..."}}
"""

In [11]:
import json
import requests
import pandas as pd

OLLAMA_URL = "http://localhost:11434/api/generate"

MODELS = [
    "ministral-3:14b",
    "nemotron-cascade-2:latest",
    "qwen3.6:35b-a3b-nvfp4",
    "gpt-oss:latest",
    "gemma4:26b"
    # añade aquí los que tengas en ollama list
]


TESTS = [
    {
        "name": "general",
        "en": "Artificial intelligence is changing the way companies design products, support customers, and automate internal processes. However, successful adoption requires clear governance, reliable evaluation methods, and careful attention to data privacy.",
        "es": "La inteligencia artificial está cambiando la forma en que las empresas diseñan productos, atienden a los clientes y automatizan procesos internos. Sin embargo, una adopción exitosa requiere una gobernanza clara, métodos de evaluación fiables y una atención cuidadosa a la privacidad de los datos.",
    },
    {
        "name": "coding_prompt",
        "en": "Review the following Python function. Identify potential bugs, suggest improvements, and rewrite it using clearer variable names and proper error handling.",
        "es": "Revisa la siguiente función de Python. Identifica posibles errores, sugiere mejoras y reescríbela usando nombres de variables más claros y una gestión de errores adecuada.",
    },
    {
        "name": "technical",
        "en": "The application uses a local inference server with streaming responses, context caching, function calling, and retrieval augmented generation over project documentation.",
        "es": "La aplicación utiliza un servidor local de inferencia con respuestas en streaming, caché de contexto, llamadas a funciones y generación aumentada por recuperación sobre la documentación del proyecto.",
    },
]

TESTS = [
    {
        "name" : "general",
        "en_en" : prompt_en_en,
        "es_es" : prompt_es_es,
        "chino_en" :  prompt_chino_en
    },
]


def ollama_prompt_tokens(model: str, prompt: str) -> int:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 100000,
            "temperature": 0,
        },
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    response.raise_for_status()
    data = response.json()
    print(f"ollama_prompt_tokens- {data}")

    if "prompt_eval_count" not in data:
        raise RuntimeError(f"Ollama did not return prompt_eval_count: {data}")


    input_tokens = data.get("prompt_eval_count", 0)
    output_tokens = data.get("eval_count", 0)
    prompt_tokens = input_tokens + output_tokens
    thinking = data.get("thinking", "")   # puede no existir
    content = data.get("response", "")    # texto generado

    return prompt_tokens, thinking, content


rows = []

for model in MODELS:
    print(f"Modelo={model}")
    for test in TESTS:
        en_en_tokens, en_en_thinking, en_en_response = ollama_prompt_tokens(model, test["en_en"])
        print(f"Modelo={model}, en_en_tokens={en_en_tokens}, en_en_thinking={len(en_en_thinking)}, en_en_response={len(en_en_response)}")
        print(en_en_thinking)
        es_es_tokens, es_es_thinking, es_es_response = ollama_prompt_tokens(model, test["es_es"])
        print(f"Modelo={model}, es_es_tokens={es_es_tokens}, es_es_thinking={len(es_es_thinking)}, es_es_response={len(es_es_response)}")
        print(es_es_thinking)
        chino_en_tokens, chino_en_thinking, chino_en_response= ollama_prompt_tokens(model, test["chino_en"])
        print(f"Modelo={model}, chino_en_tokens={chino_en_tokens}, chino_en_thinking={len(chino_en_thinking)}, chino_en_response={len(chino_en_response)}")
        print(chino_en_thinking)

        ratio = es_es_tokens / en_en_tokens
        extra_es_pct = (ratio - 1) * 100
        print(f"Modelo={model}, extra_es_pct={extra_es_pct}")

        ratio = en_en_tokens / chino_en_tokens
        extra_chino_pct = (ratio - 1) * 100
        print(f"Modelo={model}, extra_chino_pct={extra_chino_pct}")

        rows.append({
            "runtime": "ollama",
            "model": model,
            "test": test["name"],
            "mode": "raw_generate_prompt",
            "tokens_en": en_en_tokens,
            "tokens_es": es_es_tokens,
            "tokens_chino": chino_en_tokens,
            "ratio_es_en": round(ratio, 3),
            "spanish_extra_pct": round(extra_es_pct, 1),
            "chino_extra_pct": round(extra_chino_pct, 1),
        })

df = pd.DataFrame(rows)
#print(df.to_markdown(index=False))
#df.to_csv("ollama_token_results.csv", index=False)

Modelo=ministral-3:14b
ollama_prompt_tokens- {'model': 'ministral-3:14b', 'created_at': '2026-05-03T22:29:42.732759Z', 'response': '```json\n{\n  "desc": "Onboarding HRSC00000968494 was mistakenly cancelled and reactivated, but pending tasks (assets, infrastructure, email/VPN) remain incomplete or closed incorrectly.",\n  "res": "Onboarding case was restored, but workflow deletion caused task inconsistencies. Manual intervention required for asset management and VPN issues; user advised to contact relevant teams.",\n  "rootcause": "Incorrect cancellation and workflow deletion during onboarding.",\n  "res_type": "Real",\n  "ticket_type": "Incident"\n}\n```', 'done': True, 'done_reason': 'stop', 'context': [17, 4568, 1584, 29478, 2784, 1045, 1051, 1045, 1049, 1052, 1066, 47926, 8166, 1045, 1050, 1053, 1049, 1050, 1044, 1261, 43520, 26242, 11512, 1319, 23947, 1077, 1041, 6254, 1536, 42301, 2784, 26554, 1044, 1261, 8689, 53862, 3518, 125609, 1294, 6993, 1626, 4568, 4053, 1420, 26554, 27089